In [ ]:
!pip install faiss-cpu sentence-transformers


In [ ]:
import pandas as pd
import numpy as np
import os
import faiss
from sentence_transformers import SentenceTransformer
import torch
from tqdm.notebook import tqdm

# --- 1. Kaggle Paths & Setup ---
# You need to "Add Data" and select the output from Phase 2A.
INPUT_PATH = '/kaggle/input/notebooks/minatahmasebi/2a-knowledge-base' # UPDATE THIS
OUTPUT_PATH = '/kaggle/working/'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 2. Load FAISS Index and Datasets ---
print("Loading FAISS Index and Datasets...")
index = faiss.read_index(os.path.join(INPUT_PATH, 'sarcasm_kb.faiss'))

# NOTE: Due to Kaggle's Python 3.12 update, faiss-gpu is unavailable.
# We use faiss-cpu. It is extremely fast for 1M rows.
# The heavy lifting (embeddings/model) will STILL use the GPU!
# res = faiss.StandardGpuResources()
# gpu_index = faiss.index_cpu_to_gpu(res, 0, index)

kb_df = pd.read_pickle(os.path.join(INPUT_PATH, 'kb_dataset.pkl'))
df = pd.read_pickle(os.path.join(INPUT_PATH, 'full_dataset.pkl'))

print(f"Dataset to augment: {df.shape[0]} rows")

# --- 3. Initialize Embedding Model for Queries ---
# Must be exactly the same model used in Phase 2A
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

# --- 4. Retrieval Process ---
TOP_K = 3 # Retrieve top 3 similar examples

print(f"Starting Retrieval for {df.shape[0]} rows...")
# To avoid OOM, process in batches
BATCH_SIZE = 1024
augmented_texts = []

# tqdm for progress
for start_idx in tqdm(range(0, len(df), BATCH_SIZE)):
    end_idx = min(start_idx + BATCH_SIZE, len(df))
    batch_texts = df['combined_text'].iloc[start_idx:end_idx].tolist()
    
    # 1. Embed the batch
    batch_embeddings = model.encode(batch_texts, show_progress_bar=False, convert_to_numpy=True)
    
    # 2. Search in FAISS
    # D = distances, I = indices of the nearest neighbors in kb_df
    D, I = index.search(batch_embeddings, TOP_K)
    
    # 3. Construct Augmented Text
    for i in range(len(batch_texts)):
        original_text = batch_texts[i]
        
        # Get the retrieved comments from the Knowledge Base
        retrieved_examples = []
        for idx in I[i]:
            if idx != -1: # Ensure valid index
                # We only want to append the 'comment' (reply) part of the retrieved example as context
                ret_reply = kb_df.iloc[idx]['comment']
                retrieved_examples.append(f" [SIMILAR]: {ret_reply}")
        
        # Merge Original + Retrieved
        final_text = original_text + "".join(retrieved_examples)
        augmented_texts.append(final_text)

# --- 5. Save Final Augmented Dataset ---
print("Saving augmented dataset...")
df['augmented_text'] = augmented_texts

# Drop old 'combined_text' to save space
df.drop(columns=['combined_text'], inplace=True)

df.to_pickle(os.path.join(OUTPUT_PATH, 'augmented_dataset.pkl'))
print("Phase 2B Complete! You can now move to Phase 2C for training.")

